# Use case #1 — Debug an error with an LLM
## Graph Search — multi-hop traversal

**Same incident, 3am. The operator has the fix. Now they need the blast radius.**

| | |
| --- | --- |
| `01` | *"We saw `CONFIGURATION_IS_MISSING` in the logs, what should I do?"* |
| `02` | *"Makani won't start after the deploy and the logs mention something it needed..."* |
| `03` | *"For every component: who owns it, how many docs, how many superseded?"* |
| **`04`** | *"Sapir is down. **What else breaks before I fix it, and who should I talk with?**"* |

**No section contains this answer, and no row does either.** It's assembled by walking:
`Sapir ← Kesh, Nugat ← Vello, Lomi ← Tuki`, then one hop sideways to each owning team.

The postmortem in the corpus is this exact failure: 90 minutes lost debugging Tuki, then Lomi, then Nugat, while the fault sat in Sapir the whole time — *because the alert fired furthest from the cause.*

## 1. Build the graph

Etesian has **no enum of edge labels.** Labels are strings *derived* from `(source_type, dest_type, label)` by a single function, `get_edge_type()` in `ingest/edge_types.py` — so a Jira link type becomes `ticket_relates to` at runtime without any code change. The only named constants are the HR ones (`REPORTS_TO`, `MEMBER_OF`, `LIVES_IN`, `WORKS_AS`) and a `STRUCTURAL_CONTAINMENT_EDGE_LABELS` tuple that delete-cascade and ACL inheritance follow.

We copy the mechanism, with labels that fit this corpus. Every edge comes from metadata we already parse — `component`, `owner`, `depends on`, `status`.

In [1]:
from enum import StrEnum, auto

import networkx as nx
import pandas as pd

from makani import load_chunks, tokenize, bm25_index

MAX_EDGES_FOR_NODE = 100          # etesian: graph_store.py:45


class NodeType(StrEnum):          # etesian: NodeKind
    doc = auto()
    component = auto()
    team = auto()


# etesian: the containment subset that delete-cascade and ACL re-derivation follow
STRUCTURAL_CONTAINMENT_EDGE_LABELS = ("PART_OF",)


def get_edge_type(src: NodeType, dst: NodeType, label: str | None = None) -> str:
    """Labels are derived, not enumerated -- etesian's edge_types.get_edge_type()."""
    match (src, dst, label):
        case (NodeType.doc, NodeType.component, None):      return "PART_OF"
        case (NodeType.component, NodeType.component, _):   return "DEPENDS_ON"
        case (NodeType.component, NodeType.team, _):        return "OWNED_BY"
        case (NodeType.doc, NodeType.doc, "superseded"):    return "SUPERSEDED_BY"
        case (NodeType.doc, NodeType.doc, _):               return "RELATES_TO"
    raise ValueError(f"no edge type for {src} -> {dst} ({label})")


def add(G, u, v, label):
    """etesian stores edges(source, target, label) with UNIQUE(source, target, label).
    Using the label as the MultiDiGraph key gives exactly that dedup."""
    G.add_edge(u, v, key=label, label=label)


def build_graph(chunks):
    G = nx.MultiDiGraph()         # etesian subclasses this and backs it with SQLite
    docs = {}
    for c in chunks:
        docs.setdefault(c.filename, c.meta)

    for fname, meta in docs.items():
        G.add_node(fname, node_type=NodeType.doc, **meta)
        component = meta.get("component")
        if component:
            G.add_node(component, node_type=NodeType.component)
            add(G, fname, component, get_edge_type(NodeType.doc, NodeType.component))
            if meta.get("owner"):
                G.add_node(meta["owner"], node_type=NodeType.team)
                add(G, component, meta["owner"], get_edge_type(NodeType.component, NodeType.team))
            for dep in (d.strip() for d in meta.get("depends_on", "").split(",")):
                if dep and dep != "—":
                    G.add_node(dep, node_type=NodeType.component)
                    add(G, component, dep, get_edge_type(NodeType.component, NodeType.component))
        if "see " in meta.get("status", ""):
            add(G, fname, meta["status"].split("see ")[-1].strip(),
                get_edge_type(NodeType.doc, NodeType.doc, "superseded"))
    return G


chunks = load_chunks()
G = build_graph(chunks)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges\n")
print(pd.Series([k for *_, k in G.edges(keys=True)]).value_counts().to_string())

28 nodes, 29 edges

PART_OF          15
OWNED_BY          7
DEPENDS_ON        6
SUPERSEDED_BY     1


## 2. Traverse from the retrieval seeds

**The graph does not replace retrieval — it expands it.** Etesian seeds `traverse_from_seeds()` with the top 250 BM25 hits, then runs a **3-hop BFS**, stamping each node with the hop that reached it (`TraverseHop.first / second / third`). Two quirks worth copying:

- **Edges are read undirected** — `WHERE (source = :node OR target = :node)` — even though storage is directed.
- **Hubs are suppressed, not truncated.** A node with ≥100 edges returns *zero*, on the logic that a node connected to everything discriminates nothing.

So we start where notebook `01` started: BM25 over the same 71 sections.

In [2]:
def neighbours(G, n):
    """etesian reads edges regardless of direction, and suppresses hubs entirely."""
    edges = ([(v, k) for _, v, k in G.out_edges(n, keys=True)]
             + [(u, k) for u, _, k in G.in_edges(n, keys=True)])
    return [] if len(edges) >= MAX_EDGES_FOR_NODE else edges


def traverse(G, seeds, hops=3, expand=(NodeType.doc, NodeType.component)):
    """etesian's traverse_from_seeds: BFS from the retrieval seeds, coloured by hop.
    `team` nodes are attached but never expanded -- etesian does this for person."""
    hop_of = dict.fromkeys(seeds, 1)
    frontier = list(seeds)
    for hop in range(2, hops + 2):
        nxt = []
        for node in frontier:
            for neighbour, _label in neighbours(G, node):
                if neighbour in hop_of:
                    continue
                hop_of[neighbour] = hop
                if G.nodes[neighbour].get("node_type") in expand:
                    nxt.append(neighbour)
        frontier = nxt
        if not frontier:
            break
    return hop_of


question = "Sapir is down with CONFIGURATION_IS_MISSING at 3am. What else breaks and who should I talk with?"

scores = bm25_index(chunks).get_scores(tokenize(question))
seeds = list(dict.fromkeys(chunks[i].filename
                           for i in sorted(range(len(chunks)), key=lambda i: -scores[i])[:3]))
print("BM25 seeds:", seeds, "\n")

hop_of = traverse(G, seeds)
pd.DataFrame([(n, str(G.nodes[n].get("node_type")), h) for n, h in hop_of.items()],
             columns=["node", "type", "hop"]).sort_values(["hop", "type", "node"]) \
  .reset_index(drop=True)

BM25 seeds: ['sapir-config-errors.md', 'makani-architecture-overview.md', 'lomi-gateway-errors.md'] 



,node,type,hop
0,lomi-gateway-errors.md,doc,1
1,makani-architecture-overview.md,doc,1
2,sapir-config-errors.md,doc,1
3,Lomi,component,2
4,Platform,component,2
5,Sapir,component,2
6,sapir-config-errors-legacy.md,doc,2
7,Kesh,component,3
8,Nugat,component,3
9,Tuki,component,3


Three retrieved documents became **twenty nodes** — every component in the dependency chain, every owning team, and sibling docs that were never retrieved at all. `incident-2026-03-config-outage.md` and `makani-architecture-overview.md` arrive at hop 3 without matching a single query term, because they are *connected* to what did match.

**But the seeds came from BM25.** So the traversal inherits every weakness of notebook `01` — including sensitivity to how the operator happened to phrase the question.

In [3]:
COMPONENTS = {"Sapir", "Kesh", "Nugat", "Vello", "Lomi", "Tuki"}

for phrasing in ["Sapir is down with CONFIGURATION_IS_MISSING at 3am. What else breaks and who do I page?",
                 "Sapir is down with CONFIGURATION_IS_MISSING at 3am. What else breaks and who should I talk with?",
                 "Sapir is down. What else breaks?"]:
    s = bm25_index(chunks).get_scores(tokenize(phrasing))
    sd = list(dict.fromkeys(chunks[i].filename for i in sorted(range(len(chunks)), key=lambda i: -s[i])[:3]))
    reached = COMPONENTS & traverse(G, sd).keys()
    print(f"{phrasing[-34:]!r}\n   seeds:   {[f[:32] for f in sd]}"
          f"\n   reaches: {len(reached)}/6   missing: {sorted(COMPONENTS - reached) or '-'}\n")

'hat else breaks and who do I page?'
   seeds:   ['sapir-config-errors-legacy.md', 'sapir-config-errors.md', 'runbook-startup-failures.md']
   reaches: 5/6   missing: ['Tuki']

'breaks and who should I talk with?'
   seeds:   ['sapir-config-errors.md', 'makani-architecture-overview.md', 'lomi-gateway-errors.md']
   reaches: 6/6   missing: -

'Sapir is down. What else breaks?'
   seeds:   ['tuki-scheduler-errors.md', 'lomi-gateway-errors.md', 'makani-architecture-overview.md']
   reaches: 6/6   missing: -



**Same graph, same question, three phrasings — three different blast radii.** Ask *"who do I page"* and Tuki never appears: the seeds land on `sapir-config-errors-legacy.md` and the budget runs out at `doc → Sapir → {Kesh, Nugat} → {Vello, Lomi}`. Ask *"what else breaks?"* and BM25 happens to seed on Tuki's own doc, so the traversal covers everything.

Tuki is *the component that paged on-call at 02:15 in the postmortem.* Whether your 3am tooling mentions it comes down to word choice.

Etesian's traversal is hardcoded to three hops and unrolled into three literal blocks, and that is the correct design for what it does: **prompt-context expansion** — grab what's nearby, score it, fill the token budget. Approximate is fine there. It is the wrong tool for a blast radius, where a missing component *is* the failure.

## 3. Answer the question

Reachability wants no hop cap, no dependence on seeds, and exactly one edge type. This is where typed edges earn their keep.

In [4]:
DOWN = "Sapir"

depends = G.edge_subgraph([e for e in G.edges(keys=True) if e[2] == "DEPENDS_ON"])
blast = nx.ancestors(depends, DOWN)      # everything that (transitively) depends on Sapir

owner = lambda c: next((t for _, t, k in G.out_edges(c, keys=True) if k == "OWNED_BY"), "—")

pd.DataFrame([{"component": c,
               "hops from Sapir": nx.shortest_path_length(depends, c, DOWN),
               "talk to": owner(c),
               "in 3-hop BFS?": c in hop_of}
              for c in sorted(blast, key=lambda c: nx.shortest_path_length(depends, c, DOWN))],
             ).assign(**{"root cause": DOWN, "talk to for root cause": owner(DOWN)})

,component,hops from Sapir,talk to,in 3-hop BFS?,root cause,talk to for root cause
0,Kesh,1,security-team,True,Sapir,platform-team
1,Nugat,1,core-team,True,Sapir,platform-team
2,Lomi,2,edge-team,True,Sapir,platform-team
3,Vello,2,observability-team,True,Sapir,platform-team
4,Tuki,3,core-team,True,Sapir,platform-team


**Five components dark, five teams about to be woken up — and only one of them can fix it.**

The architecture doc states the rule: *"page the owning team of the deepest failed component, not the one that alerted."* That sentence is retrievable. Applying it is not: it requires knowing which component is deepest, which is a property of the graph and of nothing else.

This result does not move when you rephrase the question, because it never touched the text. Contrast the last column: whether the bounded traversal saw Tuki depended on wording; the closure does not.

### What retrieval gives you for the same question

In [5]:
from makani import show

show(chunks, scores, top_k=5)

  13.521   sapir-config-errors.md          § CONFIGURATION_IS_MISSING
  13.286   makani-architecture-overview.md § Blast radius
  13.257   lomi-gateway-errors.md          § Reading Lomi errors correctly
  12.285   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded
  10.258   incident-2026-03-config-outage. § Summary


Retrieval does better than a strawman here — `makani-architecture-overview.md § Blast radius` lands at #5, and it is genuinely the right section. Read what it actually says:

> *"A Sapir outage is a total outage — all five downstream components stay dark."*

**Five. It doesn't name them.** The names live in the dependency table in a *different section* of the same document, the owning teams live in a third place, and the escalation rule in a fourth. So the LLM receives a confident assertion that five components are affected and no way to list them — the exact section-splitting problem from notebook `01`, now load-bearing.

And be suspicious of this corpus: I wrote a section called *"Blast radius"*. Real documentation rarely has one, which makes retrieval's showing here **better** than you should expect in production, not worse.

## Takeaway

**The graph doesn't replace retrieval — it starts where retrieval stops.** Seeds come from BM25; the traversal turns 3 documents into 20 connected nodes, including documents that share no vocabulary with the question at all.

| | |
| --- | --- |
| **Labels are derived, not enumerated** | `get_edge_type(src, dst, label)` — a new Jira link type becomes `ticket_relates to` with no code change |
| **Read undirected, store directed** | `WHERE source = :n OR target = :n` |
| **Suppress hubs, don't truncate** | ≥100 edges returns *zero* — a node connected to everything discriminates nothing |
| **Bounded traversal ≠ reachability** | 3 hops is right for filling a token budget, wrong for a blast radius |
| **Typed edges are the query language** | one edge type + transitive closure = the answer; all edge types + a hop cap = context |

---

## Use case #1, closed

| Method | The question it owns | What it cannot see |
| --- | --- | --- |
| **Lexical** | a pasted log line | meaning, freshness |
| **Semantic** | the same incident, described | exact identifiers, freshness |
| **Structured** | *how many*, *which qualify* | relationships |
| **Graph** | *what else breaks, who should I talk with* | the content itself — it needs retrieval to start |

**You don't pick one.** You pick the one whose shape matches the question, and a production system runs all four — which is precisely what etesian does: BM25 seeds → graph traversal → scoring → token budget, with an LLM-written SQL query alongside it.